In [1]:
# BLOCK 0: Clean Install 

!pip uninstall -y torch torchvision torchaudio basicsr realesrgan easyocr facexlib gfpgan -q

!pip install --upgrade pip setuptools wheel -q

# PyTorch (Real-ESRGAN)
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 -q

# Basic image + data tools
!pip install opencv-python-headless pillow numpy pandas matplotlib tqdm -q

#  OCR
!pip install easyocr -q

#  Real-ESRGAN stack 
!pip install basicsr==1.4.2 realesrgan==0.3.0 facexlib==0.3.0 gfpgan==1.3.8 -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytorch-lightning 2.6.0 requires torch>=2.1.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
fastai 2.8.4 requires torch<2.9,>=1.10, which is not installed.
fastai 2.8.4 requires torchvision>=0.11, which is not installed.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.
ERROR: Could not find a version that satisfies the requirement torch==2.0.1 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2

In [2]:
# BLOCK 1: Torchvision Patch

import sys, types
import torchvision.transforms.functional as TF

# rgb_to_grayscale
fake_mod = types.ModuleType("torchvision.transforms.functional_tensor")
fake_mod.rgb_to_grayscale = TF.rgb_to_grayscale
sys.modules["torchvision.transforms.functional_tensor"] = fake_mod

print("Torchvision patch applied")


Torchvision patch applied


In [3]:
# BLOCK 2: Imports & Weights [RealESRGAN]

import os
import cv2
import numpy as np
import torch
import pandas as pd
import easyocr

from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

os.makedirs("weights", exist_ok=True)

# Download weight
if not os.path.exists("weights/RealESRGAN_x4plus.pth"):
    !wget -q -O weights/RealESRGAN_x4plus.pth \
    https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth



In [4]:
# BLOCK 3: Model Initialize

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Real-ESRGAN
model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=64,
    num_block=23,
    num_grow_ch=32,
    scale=4
)

upsampler = RealESRGANer(
    scale=4,
    model_path="weights/RealESRGAN_x4plus.pth",
    model=model,
    tile=256,
    tile_pad=10,
    pre_pad=0,
    half=False,  
    device=device
)

# EasyOCR 
reader = easyocr.Reader(['bn', 'en'], gpu=torch.cuda.is_available())

print("Models loaded")


Using device: cuda


Progress: |██████████████████████████████████████████████████| 100.0% CompleteModels loaded


In [5]:
# BLOCK 4: Preprocessing

def preprocess_for_ocr(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    denoise = cv2.fastNlMeansDenoising(gray, None, 12, 7, 21)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoise)

    _, otsu = cv2.threshold(
        enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    return otsu


In [6]:
# BLOCK 5: For confidence score
import re
import numpy as np

def extract_fields(ocr_results):
    name_confs = []
    dob_confs = []
    nid_confs = []

    nid_pattern = re.compile(r"\d{10,17}")
    full_dob_pattern = re.compile(r"\d{1,2}[-/]\d{1,2}[-/](19\d{2}|20\d{2})")
    year_pattern = re.compile(r"(19\d{2}|20\d{2})")

    for i, (bbox, text, conf) in enumerate(ocr_results):
        t = text.strip()
        compact = re.sub(r"\s+", "", t)

        # NID
        if nid_pattern.search(compact):
            nid_confs.append(conf)
            continue

        # DOB 
        if full_dob_pattern.search(compact):
            dob_confs.append(conf)
            continue

        # DOB 
        if year_pattern.search(compact):
            dob_confs.append(conf)

            if i > 0:
                prev = re.sub(r"\s+", "", ocr_results[i-1][1])
                if re.fullmatch(r"\d{1,2}", prev):
                    dob_confs.append(ocr_results[i-1][2])

            if i < len(ocr_results) - 1:
                nxt = re.sub(r"\s+", "", ocr_results[i+1][1])
                if re.fullmatch(r"\d{1,2}", nxt):
                    dob_confs.append(ocr_results[i+1][2])

            continue

        # Name confidence 
        if not re.search(r"\d", compact):
            name_confs.append(conf)

    def avg(x):
        return float(np.mean(x)) if x else 0.0

    return {
        "name_conf": float(avg(name_confs)),
        "dob_conf": float(avg(dob_confs)),
        "nid_conf": float(min(nid_confs)) if nid_confs else 0.0
    }


In [7]:
# BLOCK 6: FINAL INFERENCE
def predict_nid(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Image not readable")

    # 1) ESRGAN enhance
    img, _ = upsampler.enhance(img, outscale=2)

    # 2) Preprocess
    proc = preprocess_for_ocr(img)

    # 3) OCR
    ocr_results = reader.readtext(proc, detail=1)

    lines = []
    for _, text, conf in ocr_results:
        clean_text = text.strip()
        if clean_text:
            lines.append(clean_text)


    multiline_text = "\n".join(lines)

    # 4) Field confidence
    field_conf = extract_fields(ocr_results)

    # 5) Overall confidence
    overall_conf = (
        0.4 * field_conf["nid_conf"] +
        0.35 * field_conf["name_conf"] +
        0.25 * field_conf["dob_conf"]
    )

    return {
        "extracted_lines": lines,              
        "extracted_text": multiline_text,      
        "field_confidence": field_conf,
        "overall_confidence": round(overall_conf, 3)
    }


In [8]:
# BLOCK 7: Load Dataset Images 

import glob

image_paths = glob.glob("/kaggle/input/**/*.jpg", recursive=True)

print("Total images found:", len(image_paths))
image_paths[:5]


Total images found: 54


['/kaggle/input/datasets/prottashasaha/bd-nid-card/Image/img42.jpg',
 '/kaggle/input/datasets/prottashasaha/bd-nid-card/Image/img16.jpg',
 '/kaggle/input/datasets/prottashasaha/bd-nid-card/Image/img33.jpg',
 '/kaggle/input/datasets/prottashasaha/bd-nid-card/Image/img21.jpg',
 '/kaggle/input/datasets/prottashasaha/bd-nid-card/Image/img36.jpg']

In [9]:
# BLOCK 8: Batch Inference

from tqdm import tqdm

all_results = []

for img_path in tqdm(image_paths):
    try:
        result = predict_nid(img_path)

        all_results.append({
            "file_name": os.path.basename(img_path),
            "image_path": img_path,
            "extracted_text": result["extracted_text"],
            "name_conf": result["field_confidence"]["name_conf"],
            "dob_conf": result["field_confidence"]["dob_conf"],
            "nid_conf": result["field_confidence"]["nid_conf"],
            "overall_confidence": result["overall_confidence"]
        })

    except Exception as e:
        print("Failed:", img_path, "->", e)


  0%|          | 0/54 [00:00<?, ?it/s]

	Tile 1/25
	Tile 2/25
	Tile 3/25
	Tile 4/25
	Tile 5/25
	Tile 6/25
	Tile 7/25
	Tile 8/25
	Tile 9/25
	Tile 10/25
	Tile 11/25
	Tile 12/25
	Tile 13/25
	Tile 14/25
	Tile 15/25
	Tile 16/25
	Tile 17/25
	Tile 18/25
	Tile 19/25
	Tile 20/25
	Tile 21/25
	Tile 22/25
	Tile 23/25
	Tile 24/25
	Tile 25/25


  2%|▏         | 1/54 [00:23<21:11, 24.00s/it]

	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4


  4%|▎         | 2/54 [00:26<09:40, 11.17s/it]

...

 96%|█████████▋| 52/54 [05:30<00:10,  5.14s/it]

	Tile 1/96
	Tile 2/96
	Tile 3/96
	Tile 4/96
	Tile 5/96
	Tile 6/96
	Tile 7/96
	Tile 8/96
	Tile 9/96
	Tile 10/96
	Tile 11/96
	Tile 12/96
	Tile 13/96
	Tile 14/96
	Tile 15/96
	Tile 16/96
	Tile 17/96
	Tile 18/96
	Tile 19/96
	Tile 20/96
	Tile 21/96
	Tile 22/96
	Tile 23/96
	Tile 24/96
	Tile 25/96
	Tile 26/96
	Tile 27/96
	Tile 28/96
	Tile 29/96
	Tile 30/96
	Tile 31/96
	Tile 32/96
	Tile 33/96
	Tile 34/96
	Tile 35/96
	Tile 36/96
	Tile 37/96
	Tile 38/96
	Tile 39/96
	Tile 40/96
	Tile 41/96
	Tile 42/96
	Tile 43/96
	Tile 44/96
	Tile 45/96
	Tile 46/96
	Tile 47/96
	Tile 48/96
	Tile 49/96
	Tile 50/96
	Tile 51/96
	Tile 52/96
	Tile 53/96
	Tile 54/96
	Tile 55/96
	Tile 56/96
	Tile 57/96
	Tile 58/96
	Tile 59/96
	Tile 60/96
	Tile 61/96
	Tile 62/96
	Tile 63/96
	Tile 64/96
	Tile 65/96
	Tile 66/96
	Tile 67/96
	Tile 68/96
	Tile 69/96
	Tile 70/96
	Tile 71/96
	Tile 72/96
	Tile 73/96
	Tile 74/96
	Tile 75/96
	Tile 76/96
	Tile 77/96
	Tile 78/96
	Tile 79/96
	Tile 80/96
	Tile 81/96
	Tile 82/96
	Tile 83/96
	Tile 84/96
	

 98%|█████████▊| 53/54 [06:53<00:28, 28.43s/it]

	Tile 1/4
	Tile 2/4
	Tile 3/4
	Tile 4/4


100%|██████████| 54/54 [06:56<00:00,  7.71s/it]


In [10]:
# BLOCK 9: Save CSV

df = pd.DataFrame(all_results)

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = f"{OUTPUT_DIR}/nid_ocr_results.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("CSV saved at:", csv_path)
df.head()


CSV saved at: outputs/nid_ocr_results.csv


,file_name,image_path,extracted_text,name_conf,dob_conf,nid_conf,overall_confidence
0,img42.jpg,/kaggle/input/datasets/prottashasaha/bd-nid-ca...,গণপ্রজাতন্তট্রী বাংলাদেশ সরকার\nসরকাপ\nGovernm...,0.594032,0.976017,0.277759,0.563
1,img16.jpg,/kaggle/input/datasets/prottashasaha/bd-nid-ca...,Gqvommpলল onUuo Rী বleeলমাদেশ সরকার\nolirllo P...,0.244434,0.000000,0.000000,0.086
2,img33.jpg,/kaggle/input/datasets/prottashasaha/bd-nid-ca...,গণপ্রজাতন্ত্রী বাংলাদেশ সরকার\nGovernment of t...,0.776326,0.530997,0.740258,0.701
3,img21.jpg,/kaggle/input/datasets/prottashasaha/bd-nid-ca...,গণপ্রজাতন্ত্রী বাংলাদেশ সরকার\n(jovcmmngr! 0f ...,0.416649,0.000000,0.862003,0.491
4,img36.jpg,/kaggle/input/datasets/prottashasaha/bd-nid-ca...,গণপ্রজাতয বাংলাদেশ স্যকাব\nGouoritnuunliofU 0 ...,0.244770,0.061067,0.161632,0.166


In [11]:
# BLOCK 10: Overall Summary

print("\nAverage confidences:")
print(df[["name_conf", "dob_conf", "nid_conf", "overall_confidence"]].mean())



Average confidences:
name_conf             0.446592
dob_conf              0.409705
nid_conf              0.616856
overall_confidence    0.505519
dtype: float64


In [12]:
# BLOCK 11: OCR result to confidence score

import re
import numpy as np

def extract_fields(ocr_results):
    name_confs = []
    dob_confs = []
    nid_confs = []

    # DOB patterns
    dob_pattern1 = re.compile(r"\d{2}[-/]\d{2}[-/]\d{4}") 
    dob_pattern2 = re.compile(r"\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4}")  

    # NID pattern: 
    nid_pattern = re.compile(r"\d{10,17}")

    for bbox, text, conf in ocr_results:
        clean = text.strip()

        # DOB detection
        if dob_pattern1.search(clean) or dob_pattern2.search(clean):
            dob_confs.append(conf)
            continue

        # NID detection 
        digits_found = nid_pattern.findall(clean.replace(" ", ""))
        if digits_found:
            if any(k in clean.lower() for k in ["id", "no", "nid"]):
                nid_confs.append(conf)
                continue

            for d in digits_found:
                if len(d) >= 10:
                    nid_confs.append(conf)
                    break
            continue

        #  Name confidence 
        low = clean.lower()
        if any(k in low for k in [
            "government", "national", "card", "republic",
            "date", "birth", "id", "no"
        ]):
            continue

        # skip Bangla headers
        if any(k in clean for k in ["গণপ্রজাতন্ত্রী", "জাতীয়", "পরিচয়"]):
            continue

        name_confs.append(conf)

    def avg(x): 
        return float(np.mean(x)) if x else 0.0

    return {
        "name_conf": avg(name_confs),
        "dob_conf": avg(dob_confs),
        "nid_conf": min(nid_confs) if nid_confs else 0.0
    }


In [13]:
# BLOCK 12: Upload Image 

from ipywidgets import FileUpload
from IPython.display import display
import cv2
import numpy as np
import os

uploader = FileUpload(
    accept=".jpg,.jpeg,.png",
    multiple=False
)

display(uploader)


FileUpload(value=(), accept='.jpg,.jpeg,.png', description='Upload')

In [28]:
# BLOCK 13: Save Uploaded Image

import os

if not uploader.value:
    raise ValueError("No image uploaded")

# Handle both tuple and dict
uploaded = uploader.value[0] if isinstance(uploader.value, tuple) else list(uploader.value.values())[0]

# Name extraction 
file_name = None
if "metadata" in uploaded and "name" in uploaded["metadata"]:
    file_name = uploaded["metadata"]["name"]
elif "name" in uploaded:
    file_name = uploaded["name"]
else:
    file_name = "uploaded_nid.jpg"

# Content extraction
file_bytes = uploaded.get("content", None)
if file_bytes is None:
    raise ValueError("Upload content missing")

UPLOAD_DIR = "uploaded"
os.makedirs(UPLOAD_DIR, exist_ok=True)

image_path = os.path.join(UPLOAD_DIR, file_name)

with open(image_path, "wb") as f:
    f.write(file_bytes)

print("Image uploaded & saved at:", image_path)


Image uploaded & saved at: uploaded/img55.jpg


In [29]:
# BLOCK 14: Testing

result = predict_nid(image_path)

print("\n================ RESULT ================\n")

print("EXTRACTED TEXT:\n")
print(result["extracted_text"])

print("\nFIELD-WISE CONFIDENCE:")
for k, v in result["field_confidence"].items():
    print(f"  {k} : {round(v, 3)}")

print("\nOVERALL CONFIDENCE:")
print(result["overall_confidence"])


	Tile 1/12
	Tile 2/12
	Tile 3/12
	Tile 4/12
	Tile 5/12
	Tile 6/12
	Tile 7/12
	Tile 8/12
	Tile 9/12
	Tile 10/12
	Tile 11/12
	Tile 12/12

================ RESULT ================

EXTRACTED TEXT:

গণপ্রজাতন্ত্রী বাংলাদেশ সরকার
Government of the People's Repubiic of Bangladesh
3]|{]1
NATIONAL ID CARD
জাতীয় পরিচয় পত্র
নাম:
মোঃ জাহাঙ্গীর আলম
Name:
Md jahangir Alam
পিতা:
মোঃ শাহজাহান আলী
মাতা:
জাহানারা বেগম
Date of Birth:
15 Juri 1982
AC
|D NO:
0120802106261
L)1473

FIELD-WISE CONFIDENCE:
  name_conf : 0.615
  dob_conf : 0.573
  nid_conf : 1.0

OVERALL CONFIDENCE:
0.758
